For scatter and density plot

In [1]:
# Set directory
using CSV, DataFrames, Pandas
cd("/Users/jun/Documents/Project/Orofacial premotor circuits in the adult - RV tracing and Manipulation/Counting and quantification/roi_tables/2020_0521_test")

In [2]:
# read files
wp007 = CSV.read("roi_table_007retro_wp4_20200408.txt");
mas016 = CSV.read("016_mas2.txt");
genio4 = CSV.read("024retro_genio4.txt");

file_name = [:wp007 :mas016 :genio4];

In [3]:
# convert roi_table coordinates into Allen CCF coordinates

for i = 1:length(file_name);
    eval(file_name[i]).AP_location = -(eval(file_name[i]).AP_location*100);
    eval(file_name[i]).ML_location = eval(file_name[i]).ML_location*100;
    eval(file_name[i]).DV_location = eval(file_name[i]).DV_location*100;
end

In [4]:
# find rostral and caudal ends

nucleus = "IRN"

rostral = [];
caudal = [];
for i = 1:length(file_name);
max = maximum(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location);
min = minimum(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location);
push!(caudal, max);
push!(rostral, min);
end

rostral_end = convert(Int64, floor(maximum(rostral))) + 540; # AP corrdinate of Bregma in Allen CCF is 540
caudal_end = convert(Int64, floor(minimum(caudal))) + 540;

┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[4]:7
└ @ Core ./In[4]:7
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[4]:9
└ @ Core ./In[4]:9


In [5]:
# brain outlines
# coronal
using NPZ
atlas = npzread("/Users/jun/Documents/MATLAB/Allen/annotation_volume_10um_by_index.npy");

# coronal outline
NumberOfMerge = convert(Int64, floor(caudal_end - rostral_end));

AP,DV,ML = size(atlas)
coronal = Array{Float16, 3}(undef, DV, ML, NumberOfMerge);
using Images
for i =1:NumberOfMerge
    coronal[:,:,i] = canny(ifelse.(convert(Array{Float16}, atlas[floor(rostral_end) + i,:,:]) .<=1, 0, 1),(0.01,0.0));
end
coronal_merge = sum(coronal, dims = 3);
coronal_merge = convert(Array{Float64,2},coronal_merge[:,:,1]);


In [6]:
# horizontal outline
NumberOfMerge = 100 # arbitrary value.
AP,DV,ML = size(atlas)
horizontal = Array{Float16, 3}(undef, AP, ML, NumberOfMerge);

using Images
for i =1:NumberOfMerge
    horizontal[:,:,i] = canny(ifelse.(convert(Array{Float16}, atlas[:,550 + i,:])
     .<=1, 0, 1),(0.01,0.0));
     # atlas[:, x + i,:]) x = arbitary value. find a value that gives a good outline
end

horizontal_merge = sum(horizontal, dims = 3);
horizontal_merge = convert(Array{Float64,2}, horizontal_merge[:,:,1]);
horizontal_merge = horizontal_merge[840:1320,:,:]; # crop in arbitary AP range

In [16]:
# test visualization coronal
# using PyCall, PyPlot
# fig, ax = plt.subplots(1,1, figsize=(10,10))
# ax.imshow(coronal_merge[:,:,1], extent =[-570, 570, 800, 0] ,cmap="binary", zorder = 0); #
# plt.axis("equal");

In [19]:
# test visualization horizontal
# fig, ax = plt.subplots(1,1, figsize=(10,5))
# ax.imshow(horizontal_merge[:,:,1], extent =[-570, 570, 780, 310] ,cmap="binary", zorder = 0);
# plt.axis("equal");
# ax.set(ylim=(780, 300))

In [18]:
# test visualization sagital
# fig, ax = plt.subplots(1,1, figsize=(10,10))
# ax.imshow(sagittal_merge[:,:,1], cmap="binary", zorder = 0);
# plt.axis("equal");

In [7]:
# coronal
using PyCall, PyPlot
sns = pyimport("seaborn")
pygui(true)
plt.style.use("gadfly") # plot with Gadfly style

# coronal
# scatter plot
# c = ["#00BCFD" "#D3C93A" "#FF62A4"]
c = ["#FF62A4" "#D3C93A" "#00BCFD"]
cmap = ["PuRd" "Wistia" "Blues"] # PuRd
i = 1
j = 1
k = 1
while i <= length(file_name);
        # scatter plot
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,10))
        ax.imshow(coronal_merge[:,:,1], extent =[-570, 570, 800, 0] ,cmap="binary", zorder = 0); #
        plt.axis("equal");

        ax.scatter(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].ML_location,
        eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].DV_location,
        c = c[i],  s =0.5, zorder = 1)
    
        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string((file_name[i]), "_"  ,nucleus,  "_coronal.png"), dpi = 600, format = "png")

        # density plot
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,10))
        ax.imshow(coronal_merge[:,:,1], extent =[-570, 570, 800, 0] ,cmap="binary", zorder = 0); #
        plt.axis("equal");

        ax = sns.kdeplot(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].ML_location,
        eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].DV_location,
        cmap = cmap[i], n_levels =4, bw = 18, linewidths = 1.5, zorder = 1)
        
        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string((file_name[i]), "_"  ,nucleus,  "_coronal_density.png"), dpi = 600, format = "png")
        global i = i + 1

    if i == length(file_name) + 1;
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,10))
        ax.imshow(coronal_merge[:,:,1], extent =[-570, 570, 800, 0] ,cmap="binary", zorder = 0); #
        plt.axis("equal");
        while j <= length(file_name)
            
            ax.scatter(eval(file_name[j])[eval(file_name[j])[:acronym].== eval(nucleus),:].ML_location,
            eval(file_name[j])[eval(file_name[j])[:acronym].== eval(nucleus),:].DV_location,
            c = c[j],  s =0.5, zorder = 1)
            
            global j = j + 1
            
        if j == length(file_name) + 1

            ax.grid()
            ax.xaxis.set_major_formatter(plt.NullFormatter())
            ax.yaxis.set_major_formatter(plt.NullFormatter())

            savefig(string("merge", "_", nucleus, "_coronal.png"), dpi = 600, format = "png")
            close()
            fig, ax = plt.subplots(1,1, figsize=(10,10))
            ax.imshow(coronal_merge[:,:,1], extent =[-570, 570, 800, 0] ,cmap="binary", zorder = 0); #
            plt.axis("equal");
            while k <= length(file_name)
                # density plot
                ax = sns.kdeplot(eval(file_name[k])[eval(file_name[k])[:acronym].== eval(nucleus),:].ML_location,
                eval(file_name[k])[eval(file_name[k])[:acronym].== eval(nucleus),:].DV_location,
                cmap = cmap[k], n_levels =4, bw = 18, linewidths = 1.5, zorder = 1)
                global k = k + 1
                
                    if k == length(file_name) + 1

                    ax.grid()
                    ax.xaxis.set_major_formatter(plt.NullFormatter())
                    ax.yaxis.set_major_formatter(plt.NullFormatter())

                    savefig(string("merge", "_"  ,nucleus,  "_coronal_density.png"), dpi = 600, format = "png")
                    close()
end
end
end
end
end
end

Illegal line #617
	"view rawmatplotlibrc hosted with ❤ by GitHub
"
	in file "/Users/jun/.matplotlib/stylelib/gadfly.mplstyle"
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[7]:22
└ @ Core ./In[7]:22
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[7]:22
└ @ Core ./In[7]:22
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[7]:38
└ @ Core ./In[7]:38
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[7]:38
└ @ Core ./In[7]:38
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[7]:56
└ @ Core ./In[7]:56
┌ Warning: `getindex(df::DataFrame, col_ind::Colu

In [8]:
# horizontal
using PyCall, PyPlot
sns = pyimport("seaborn")
pygui(true)
plt.style.use("gadfly") # plot with Gadfly style

# coronal
# scatter plot
c = ["#FF62A4" "#D3C93A" "#00BCFD"]
cmap = ["PuRd" "Wistia" "Blues"] # PuRd
i = 1
j = 1
k = 1
while i <= length(file_name);
        # scatter plot
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,5))
        ax.imshow(horizontal_merge[:,:,1], extent =[-570, 570, 780, 310] ,cmap="binary", zorder = 0);
        plt.axis("equal");
        ax.set(ylim=(780, 300))

        ax.scatter(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].ML_location,
        eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location,
        c = c[i],  s =0.5, zorder = 1)
    
        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string((file_name[i]), "_"  ,nucleus,  "_horizontal.png"), dpi = 600, format = "png")

        # density plot
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,5))
        ax.imshow(horizontal_merge[:,:,1], extent =[-570, 570, 780, 310] ,cmap="binary", zorder = 0);
        plt.axis("equal");
        ax.set(ylim=(780, 300))

        ax = sns.kdeplot(eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].ML_location,
        eval(file_name[i])[eval(file_name[i])[:acronym].== eval(nucleus),:].AP_location,
        cmap = cmap[i], n_levels =4, bw = 18, linewidths = 1.5, zorder = 1)
        
        ax.grid()
        ax.xaxis.set_major_formatter(plt.NullFormatter())
        ax.yaxis.set_major_formatter(plt.NullFormatter())

        savefig(string((file_name[i]), "_"  ,nucleus,  "_horizontal_density.png"), dpi = 600, format = "png")
        global i = i + 1

    if i == length(file_name) + 1;
        close()
        fig, ax = plt.subplots(1,1, figsize=(10,5))
        ax.imshow(horizontal_merge[:,:,1], extent =[-570, 570, 780, 310] ,cmap="binary", zorder = 0);
        plt.axis("equal");
        ax.set(ylim=(780, 300))
        
        while j <= length(file_name)
            
            ax.scatter(eval(file_name[j])[eval(file_name[j])[:acronym].== eval(nucleus),:].ML_location,
            eval(file_name[j])[eval(file_name[j])[:acronym].== eval(nucleus),:].AP_location,
            c = c[j],  s =0.5, zorder = 1)
            
            global j = j + 1
            
        if j == length(file_name) + 1

            ax.grid()
            ax.xaxis.set_major_formatter(plt.NullFormatter())
            ax.yaxis.set_major_formatter(plt.NullFormatter())

            savefig(string("merge", "_", nucleus, "_horizontal.png"), dpi = 600, format = "png")
            close()
            fig, ax = plt.subplots(1,1, figsize=(10,5))
            ax.imshow(horizontal_merge[:,:,1], extent =[-570, 570, 780, 310] ,cmap="binary", zorder = 0);
            plt.axis("equal");
            ax.set(ylim=(780, 300))
                
            while k <= length(file_name)
                # density plot
                ax = sns.kdeplot(eval(file_name[k])[eval(file_name[k])[:acronym].== eval(nucleus),:].ML_location,
                eval(file_name[k])[eval(file_name[k])[:acronym].== eval(nucleus),:].AP_location,
                cmap = cmap[k], n_levels =4, bw = 18, linewidths = 1.5, zorder = 1)
                global k = k + 1
                
                    if k == length(file_name) + 1

                    ax.grid()
                    ax.xaxis.set_major_formatter(plt.NullFormatter())
                    ax.yaxis.set_major_formatter(plt.NullFormatter())

                    savefig(string("merge", "_"  ,nucleus,  "_horizontal_density.png"), dpi = 600, format = "png")
                    close()
end
end
end
end
end
end

┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[8]:22
└ @ Core ./In[8]:22
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[8]:22
└ @ Core ./In[8]:22
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[8]:39
└ @ Core ./In[8]:39
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[8]:39
└ @ Core ./In[8]:39
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[8]:59
└ @ Core ./In[8]:59
┌ Warning: `getindex(df::DataFrame, col_ind::ColumnIndex)` is deprecated, use `df[!, col_ind]` instead.
│   caller = top-level scope at In[8]:59
└ @ Core ./In[8]:59
┌ Warning: